In [53]:
import numpy as np
import json
import fastai
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


from pathlib import Path

import torch
from fastai.text.all import *

from GrooveModel import Datasets
from GrooveModel.Utils import DNAValue

In [54]:
# Paths
BASE_PATH = Path.cwd().parent
DATA_PATH = BASE_PATH / 'Data'
DNA_PATH = DATA_PATH / 'dnas.json'
MODEL_PATH = BASE_PATH / 'GrooveModel'
XLSTM_PATH = MODEL_PATH / 'xlstm'

In [55]:
# Dataset Config
ds_conf = {
    'dna_path': DNA_PATH
    # ADD more params like subset, training/validation/testing,...
}

In [56]:
# Dataset
dna_ds = Datasets.DNADataset(ds_conf, 'train')
len(dna_ds)

622

In [57]:
first_track = dna_ds[0]
for dna_unit in first_track['DNAUnits']:
    print(DNAValue.dna_to_instruments(dna_unit['Value']))

[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
['Snare']
['Snare']
['Kick']
['Kick']
['Kick']
['Kick', 'Toms']
['Snare']
['Snare']
['Kick']
['Kick']
['Kick']
['Kick', 'Toms']
['Snare']
[]
[]
[]
['Kick', 'Crash']
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
[]
['Toms']
['Kick', 'Toms']
['Kick']
[]
['Kick']
[]
['Snare']
[]
['Kick']
[]
['Toms']
[]
['Kick']


In [58]:
# separate fill/beat

In [59]:
embedding_layer = nn.Embedding(10, 3, sparse=True)

In [60]:
output = embedding_layer(torch.tensor([1, 2, 3, 4]))
output

tensor([[ 0.6443, -1.3927, -1.3489],
        [ 0.1595, -0.6877, -1.2220],
        [-0.6608,  0.0600, -0.1301],
        [-0.2402, -0.4931, -0.0577]], grad_fn=<EmbeddingBackward0>)

In [61]:
# dot product of each row vector

np.dot(output[0].detach().numpy(), output[1].detach().numpy())

np.float32(2.7088766)

In [62]:
np.dot(output[0].detach().numpy(), output[2].detach().numpy())

np.float32(-0.333824)

In [63]:
np.dot(output[0].detach().numpy(), output[3].detach().numpy())

np.float32(0.6097731)

In [64]:
from GrooveModel.Utils.DNAOffset import normalize_offset_ticks
from GrooveModel.Utils.DNAVelocity import normalize_velocity
from dataclasses import dataclass
from GrooveModel.Utils.DNAValue import InstrumentValues

from dataclasses import dataclass
from typing import List
import torch

@dataclass
class DNAToken:
    Instrument: int
    Velocity: int
    GridUnit: int
    GridUnitOffset: int
    GridFactor: int
    Bpm: int
    Numerator: int
    Denominator: int
    Measures: int
    TicksPerQuarter: int
    Source: string

def extract_dna_tokens(song_json: dict, trim_leading_empty_measures=True) -> List[DNAToken]:
    tokens = []

    # TODO: Groove/Fill Split

    bpm = song_json.get('Bpm', 120)
    numerator = song_json.get('Numerator', 4)
    denominator = song_json.get('Denominator', 4)
    ticks_per_quarter = song_json.get('TicksPerQuarterNote', 480)
    grid_factor = song_json.get('GridFactor', 1)
    measures = song_json.get('Measures', 1)
    source = song_json.get('DNA_ID', 'Unknown')

    dna_units = song_json.get('DNAUnits', [])

    if trim_leading_empty_measures:
        empty_measures = 0
        measure_size = grid_factor * numerator  # units per measure

        for measure_index in range(0, len(dna_units), measure_size):
            measure = dna_units[measure_index:measure_index + measure_size]

            # Check if the entire measure is empty
            if all(unit['IsEmpty'] for unit in measure):
                empty_measures += 1
            else:
                break  # Stop trimming once we find a non-empty measure

        dna_units = dna_units[empty_measures * measure_size:]

    grid_unit = 0
    for dna_unit in dna_units:
        value = dna_unit.get('Value', 0)
        instruments = DNAValue.dna_to_dna_instruments_list(dna_unit['Value'])
        velocities_per_value_part = dna_unit.get('VelocityPerValuePart', {})
        offset_per_value_part = dna_unit.get('OffsetTicksPerValuePart', {})

        token = None

        for instrument in instruments:
            velocity = normalize_velocity(velocities_per_value_part[str(instrument)])
            offset = normalize_offset_ticks(offset_per_value_part[str(instrument)], ticks_per_quarter)

            token = DNAToken(
            Instrument=instrument,
            Velocity=velocity,
            GridUnit=grid_unit,
            GridUnitOffset=offset,
            Bpm=bpm,
            Numerator=numerator,
            Denominator=denominator,
            GridFactor=grid_factor,
            Measures=measures,
            TicksPerQuarter=ticks_per_quarter,
            Source=source,
            )

        if token is None:
            # empty grid unit
            token = DNAToken(
            Instrument=0,
            Velocity=0,
            GridUnit=grid_unit,
            GridUnitOffset=0,
            Bpm=bpm,
            Numerator=numerator,
            Denominator=denominator,
            GridFactor=grid_factor,
            Measures=measures,
            TicksPerQuarter=ticks_per_quarter,
            Source=source,
            )
        tokens.append(token)
        grid_unit += 1
        grid_unit = grid_unit % (grid_factor * numerator) # Grid unit in Measure
        # TODO: maybe absolute gridunit would be better

    return tokens

tokens = extract_dna_tokens(dna_ds[1])
print(tokens[0])

DNAToken(Instrument=32, Velocity=127, GridUnit=0, GridUnitOffset=480, GridFactor=4, Bpm=160, Numerator=4, Denominator=4, Measures=1, TicksPerQuarter=480, Source='fbo_10_10')


In [65]:
def test_extract_dna_tokens_trims_leading_empty_measures():
    # 4/4 time, 4 units per measure
    dna_units = [
        # Measure 1 - Empty
        {'IsEmpty': True, 'Value': 0, 'VelocityPerValuePart': {}, 'OffsetTicksPerValuePart': {}} for _ in range(4)
    ] + [
        # Measure 2 - Not Empty
        {'IsEmpty': False, 'Value': 1, 'VelocityPerValuePart': {'1': 1}, 'OffsetTicksPerValuePart': {'1': 0}}
    ] + [
        {'IsEmpty': True, 'Value': 0, 'VelocityPerValuePart': {}, 'OffsetTicksPerValuePart': {}} for _ in range(3)
    ]

    song_json = {
        'Bpm': 120,
        'Numerator': 4,
        'Denominator': 4,
        'TicksPerQuarterNote': 480,
        'GridFactor': 1,
        'DNAUnits': dna_units,
        'Measures': 2
    }

    tokens = extract_dna_tokens(song_json, trim_leading_empty_measures=True)

    assert len(tokens) == 4, f"Expected 4 tokens after trimming, got {len(tokens)}"
    assert tokens[0].Instrument == 1, "First token should be the non-empty instrument"
    print("Test 1 passed: Leading empty measure trimmed correctly.")


def test_extract_dna_tokens_no_trimming():
    # Song starts immediately with a note
    dna_units = [
        {'IsEmpty': False, 'Value': 1, 'VelocityPerValuePart': {'1': 1}, 'OffsetTicksPerValuePart': {'1': 0}}
    ] + [
        {'IsEmpty': True, 'Value': 0, 'VelocityPerValuePart': {}, 'OffsetTicksPerValuePart': {}} for _ in range(7)
    ]

    song_json = {
        'Bpm': 120,
        'Numerator': 4,
        'Denominator': 4,
        'TicksPerQuarterNote': 480,
        'GridFactor': 1,
        'DNAUnits': dna_units,
        'Measures': 2
    }

    tokens = extract_dna_tokens(song_json, trim_leading_empty_measures=True)

    assert len(tokens) == 8, f"Expected 8 tokens, got {len(tokens)}"
    assert tokens[0].Instrument == 1, "First token should be the non-empty instrument"
    print("Test 2 passed: No trimming when song starts with note.")


def test_extract_dna_tokens_partial_empty_measure():
    # First measure has some non-empty units → should not be trimmed
    dna_units = [
        {'IsEmpty': True, 'Value': 0, 'VelocityPerValuePart': {}, 'OffsetTicksPerValuePart': {}},
        {'IsEmpty': False, 'Value': 1, 'VelocityPerValuePart': {'1': 1}, 'OffsetTicksPerValuePart': {'1': 0}},
        {'IsEmpty': True, 'Value': 0, 'VelocityPerValuePart': {}, 'OffsetTicksPerValuePart': {}},
        {'IsEmpty': True, 'Value': 0, 'VelocityPerValuePart': {}, 'OffsetTicksPerValuePart': {}}
    ] + [
        {'IsEmpty': True, 'Value': 0, 'VelocityPerValuePart': {}, 'OffsetTicksPerValuePart': {}} for _ in range(4)
    ]

    song_json = {
        'Bpm': 120,
        'Numerator': 4,
        'Denominator': 4,
        'TicksPerQuarterNote': 480,
        'GridFactor': 1,
        'DNAUnits': dna_units,
        'Measures': 2
    }

    tokens = extract_dna_tokens(song_json, trim_leading_empty_measures=True)

    assert len(tokens) == 8, f"Expected 8 tokens, got {len(tokens)}"
    assert tokens[0].Instrument == 0, "First token should be a rest"
    print("Test 3 passed: Measure with any note is kept.")


def test_extract_dna_tokens_all_empty():
    # All measures are empty → should trim all
    dna_units = [
        {'IsEmpty': True, 'Value': 0, 'VelocityPerValuePart': {}, 'OffsetTicksPerValuePart': {}} for _ in range(8)
    ]

    song_json = {
        'Bpm': 120,
        'Numerator': 4,
        'Denominator': 4,
        'TicksPerQuarterNote': 480,
        'GridFactor': 1,
        'DNAUnits': dna_units,
        'Measures': 2
    }

    tokens = extract_dna_tokens(song_json, trim_leading_empty_measures=True)

    assert len(tokens) == 0, f"Expected 0 tokens after trimming, got {len(tokens)}"
    print("Test 4 passed: All empty measures correctly trimmed.")

test_extract_dna_tokens_trims_leading_empty_measures()
test_extract_dna_tokens_no_trimming()
test_extract_dna_tokens_partial_empty_measure()
test_extract_dna_tokens_all_empty()

# TODO Move to unit tests

Test 1 passed: Leading empty measure trimmed correctly.
Test 2 passed: No trimming when song starts with note.
Test 3 passed: Measure with any note is kept.
Test 4 passed: All empty measures correctly trimmed.


In [66]:
for song in dna_ds:
    if not song['DNAUnits'][0]['IsEmpty']:
        print('not empty')

In [67]:
len(tokens)

255

In [68]:
all_tokens = []
for song in dna_ds:
    all_tokens.extend(extract_dna_tokens(song, trim_leading_empty_measures=True))

len(all_tokens)

89425

In [69]:
# distribution of all numerators
# Extract the fields into lists
numerators = [token.Numerator for token in all_tokens]
denominators = [token.Denominator for token in all_tokens]
ticks_per_quarter = [token.TicksPerQuarter for token in all_tokens]
grid_factors = [token.GridFactor for token in all_tokens]
bpms = [token.Bpm for token in all_tokens]

# Count occurrences
numerator_counts = Counter(numerators)
denominator_counts = Counter(denominators)
ticks_per_quarter_counts = Counter(ticks_per_quarter)
grid_factor_counts = Counter(grid_factors)
bpm_counts = Counter(bpms)

# Print results
print("Numerator Counts:", numerator_counts)
print("Denominator Counts:", denominator_counts)
print("Ticks Per Quarter Counts:", ticks_per_quarter_counts)
print("Grid Factor Counts:", grid_factor_counts)
print("BPM Counts:", bpm_counts)

Numerator Counts: Counter({4: 84878, 3: 4547})
Denominator Counts: Counter({4: 89425})
Ticks Per Quarter Counts: Counter({480: 89425})
Grid Factor Counts: Counter({4: 47050, 6: 42375})
BPM Counts: Counter({180: 17637, 120: 12331, 140: 10762, 115: 10100, 100: 9826, 150: 9495, 200: 8933, 160: 5794, 142: 4547})


In [70]:
TIME_SIGNATURE_LOOKUP = {
    (4, 4): 0,
    (3, 4): 1,
    (6, 8): 2,
    (2, 4): 3,
    (2, 2): 4,
    (5, 4): 5,
    (7, 8): 6,
    (9, 8): 7,
    (12, 8): 8,
    (3, 8): 9,
    (6, 4): 10,
    (3, 2): 11
    # Add more as needed
}

# Optional reverse lookup
ID_TO_TIME_SIGNATURE = {v: k for k, v in TIME_SIGNATURE_LOOKUP.items()}

TIME_SIGNATURES_SIZE = len(TIME_SIGNATURE_LOOKUP)

UNKNOWN_TIME_SIGNATURE_ID = TIME_SIGNATURES_SIZE



def encode_time_signature(numerator: int, denominator: int) -> int:
    """
    Encodes the time signature into an ID, starting denominators at 2.
    """
    if denominator < 2:
        raise ValueError("Denominator must be >= 2.")
    return TIME_SIGNATURE_LOOKUP.get((numerator, denominator), UNKNOWN_TIME_SIGNATURE_ID)

def decode_time_signature(time_id: int) -> (int, int):
    """
    Decodes the time signature ID back to (numerator, denominator).
    """
    if time_id not in ID_TO_TIME_SIGNATURE:
        raise ValueError(f"Unknown time signature ID: {time_id}")
    return ID_TO_TIME_SIGNATURE[time_id]

encode_time_signature(3,2)

11

In [71]:

class GridFactors(IntEnum):
    Quarter = 1
    Eighth = 2
    Sixteenth = 4
    EighthTriplet = 3
    SixteenthTriplet = 6

class RemappedGridFactors(IntEnum):
    Quarter = 0
    Eighth = 1
    Sixteenth = 2
    EighthTriplet = 3
    SixteenthTriplet = 4

GRID_FACTORS_SIZE = sum([value.value for value in GridFactors]) + 1

def dna_grid_factor_to_remapped_factor(grid_factor) -> RemappedGridFactors:
    dna_grid_factor = GridFactors(grid_factor).name
    return RemappedGridFactors[dna_grid_factor]

In [72]:
from GrooveModel.Utils.DNAValue import dna_instrument_to_remapped_value


# REMAPPING
# remap instruments to 0-6 ids
for token in all_tokens:
    token.Instrument = dna_instrument_to_remapped_value(token.Instrument)
    token.GridFactor = dna_grid_factor_to_remapped_factor(token.GridFactor)

In [73]:
from GrooveModel.Utils.DNAValue import DNA_VALUE_SIZE
from GrooveModel.Utils.DNAVelocity import VELOCITY_SIZE
from GrooveModel.Utils.DNAOffset import OFFSET_TICKS_SIZE


#TODO: BPM normalization (will do 0 - 300 for right now)
# vocab sizes
instruments_vocab_size = DNA_VALUE_SIZE
velocities_vocab_size = VELOCITY_SIZE
offsets_vocab_size = OFFSET_TICKS_SIZE
bpm_vocab_size = 300
time_signature_vocab_size = TIME_SIGNATURES_SIZE
grid_factor_vocab_size = GRID_FACTORS_SIZE


# embedding dimensions
instrument_embedding_dim = 4 # 7 values
velocity_embedding_dim = 16 # 128 values
offset_embedding_dim = 32 # 960 values
time_signature_embedding_dim = 6 # 40 values
grid_embedding_dim = 4 # 5 values
bpm_embedding_dim = 24 # 300 values
position_embedding_dim = 16 # positional embedding is concatenated

embedding_config = \
{
    'instruments_vocab_size': instruments_vocab_size,
    'instrument_embedding_dim': instrument_embedding_dim,
    'velocities_vocab_size': velocities_vocab_size,
    'velocity_embedding_dim': velocity_embedding_dim,
    'offsets_vocab_size': offsets_vocab_size,
    'offset_embedding_dim': offset_embedding_dim,
    'time_signature_vocab_size': time_signature_vocab_size,
    'time_signature_embedding_dim': time_signature_embedding_dim,
    'grid_factor_vocab_size': grid_factor_vocab_size,
    'grid_embedding_dim': grid_embedding_dim,
    'bpm_vocab_size': bpm_vocab_size,
    'bpm_embedding_dim': bpm_embedding_dim,
    'positional_embedding_dim' : position_embedding_dim,
}


In [74]:
class DNAContentEmbedding(nn.Module):
    def __init__(self, config):
        super(DNAContentEmbedding, self).__init__()

        self.instrument_embedding = nn.Embedding(config['instruments_vocab_size'], config['instrument_embedding_dim'])
        self.velocity_embedding = nn.Embedding(config['velocities_vocab_size'], config['velocity_embedding_dim'])
        self.offset_embedding = nn.Embedding(config['offsets_vocab_size'], config['offset_embedding_dim'])
        self.time_signature_embedding = nn.Embedding(config['time_signature_vocab_size'], config['time_signature_embedding_dim'])
        self.grid_embedding = nn.Embedding(config['grid_factor_vocab_size'], config['grid_embedding_dim'])
        self.bpm_embedding = nn.Embedding(config['bpm_vocab_size'], config['bpm_embedding_dim'])

        self.content_embedding_dim = (config['instrument_embedding_dim'] +
                                      config['velocity_embedding_dim'] +
                                      config['offset_embedding_dim'] +
                                      config['time_signature_embedding_dim'] +
                                      config['grid_embedding_dim'] +
                                      config['bpm_embedding_dim'])

    def forward(self, token):
        instrument_token_embedding = self.instrument_embedding(torch.tensor([token.Instrument]))
        velocity_token_embedding = self.velocity_embedding(torch.tensor([token.Velocity]))
        offset_token_embedding = self.offset_embedding(torch.tensor([token.GridUnitOffset]))
        time_signature_token_embedding = self.time_signature_embedding(torch.tensor([encode_time_signature(token.Numerator, token.Denominator)]))
        grid_token_embedding = self.grid_embedding(torch.tensor([token.GridFactor]))
        bpm_token_embedding = self.bpm_embedding(torch.tensor([token.Bpm]))

        token_embedding = torch.cat([
            instrument_token_embedding,
            velocity_token_embedding,
            offset_token_embedding,
            time_signature_token_embedding,
            grid_token_embedding,
            bpm_token_embedding
        ], dim=-1).squeeze(0)

        return token_embedding

In [75]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, positional_embedding_dim):
        super(SinusoidalPositionalEncoding, self).__init__()
        self.positional_embedding_dim = positional_embedding_dim

    def forward(self, grid_unit):
        pe = torch.zeros(self.positional_embedding_dim)
        position = grid_unit
        for i in range(0, self.positional_embedding_dim, 2):
            div_term = math.exp(-math.log(10000.0) * i / self.positional_embedding_dim)
            pe[i] = math.sin(position * div_term)
            if i + 1 < self.positional_embedding_dim:
                pe[i + 1] = math.cos(position * div_term)
        return pe

In [76]:
class DNAEmbedding(nn.Module):
    def __init__(self, config):
        super(DNAEmbedding, self).__init__()

        self.content_embedding = DNAContentEmbedding(config)
        self.positional_encoding = SinusoidalPositionalEncoding(config['positional_embedding_dim'])

        self.total_embedding_dim = self.content_embedding.content_embedding_dim + config['positional_embedding_dim']

    def forward(self, token):
        content_embedding = self.content_embedding(token)
        positional_embedding = self.positional_encoding(token.GridUnit)

        final_embedding = torch.cat([content_embedding, positional_embedding], dim=-1)
        return final_embedding

In [77]:
embedder = DNAEmbedding(embedding_config)
embedder

DNAEmbedding(
  (content_embedding): DNAContentEmbedding(
    (instrument_embedding): Embedding(64, 4)
    (velocity_embedding): Embedding(128, 16)
    (offset_embedding): Embedding(960, 32)
    (time_signature_embedding): Embedding(12, 6)
    (grid_embedding): Embedding(17, 4)
    (bpm_embedding): Embedding(300, 24)
  )
  (positional_encoding): SinusoidalPositionalEncoding()
)

In [78]:
try:
    for token in all_tokens:
        embedder(token)

except Exception as e:
    print(token)
    print(e)

In [80]:
embedder(all_tokens[1])

tensor([-5.0166e-01, -3.3922e-01,  7.8672e-01,  2.1722e+00, -3.6265e-01,
         9.4777e-01,  6.2350e-01, -3.9922e-01,  8.5223e-01,  9.0577e-01,
        -5.6127e-01,  2.3872e+00, -6.4602e-01,  2.7519e-01,  5.9209e-01,
         9.5444e-01, -1.6858e+00, -3.2463e-01, -1.8671e-01,  2.6205e-01,
        -5.2827e-02,  2.6761e-02, -9.6762e-01,  4.7276e-02,  8.0480e-02,
        -6.1396e-01,  1.4717e-01, -1.9120e+00, -8.0429e-01,  5.6550e-01,
         1.1501e+00,  9.3681e-01, -1.2764e+00, -1.3743e+00,  6.8984e-01,
        -4.9899e-01,  5.0024e-04,  1.2315e+00, -9.8272e-01, -2.2578e-01,
         4.8065e-01, -4.0985e-01,  1.0969e+00,  7.7011e-03,  1.2311e+00,
         8.4247e-01, -6.0839e-01, -3.1611e-02,  5.6588e-01, -2.4556e+00,
        -6.9662e-01,  1.0696e-01, -8.1890e-02,  1.3569e+00, -1.2204e+00,
         5.1106e-02,  1.9375e+00,  6.7719e-02, -5.8604e-01,  1.1044e+00,
        -3.4370e-01, -1.5821e-01, -2.4721e-02, -1.2152e+00,  9.0883e-01,
         1.7025e-01, -8.6974e-01,  4.2330e-01,  3.6

In [79]:
# split beat/fill ? // OPEN
# create embeddings for all required dna values // DONE
# conat them // DONE
# positional embedding according to gridunit (beat position) // DONE
# feed to xlstm
# split output tensor
# reconstruct dna unit

# Rewrite all into python files




1. Data Preprocessing will require a step of preforming the file read layout (mid + text)
2. Make a class for preparing midi files and their meta info to dna-layout
3. Make class abstract and define specific readers for each source (foundational vs. lmd vs. basti ,...)
4. Run json extractor
5. Create dataset with json
6. create tokens
7. embed tokens
8. ...


# DOCUMENT AFTER TEST!!!!!!!!!!!